In [5]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

In [6]:
load_dotenv()

True

In [ ]:
model = ChatGroq(
    model="llama-3.1-8b-instant",   # free + fast model
    temperature=0
)

In [8]:
class Blog_state(TypedDict):
    title : str
    outline: str
    blog: str
    score: int

In [ ]:
def outline_draft(state: Blog_state) -> Blog_state:
    title = state['title']

    prompt = f"Generate an detailed outline of a blog to be created on the topic - {title}"

    response = model.invoke(prompt).content

    state['outline'] = response

    return state


def Blog_draft(state: Blog_state) -> Blog_state:
    title = state['title']
    outline = state['outline']

    prompt = f"Generate an detailed blog on the topic - {title} using following outline \n {outline}"

    response = model.invoke(prompt).content

    state['blog'] = response

    return state


def Evaluator(state: Blog_state) -> Blog_state:
    blog = state['blog']
    outline = state['outline']

    prompt = f"On rate of 1 to 10, Rate the similarity of blog- {blog} with the outline - {outline}"

    response = model.invoke(prompt).content

    state['score'] = response

    return state


In [10]:
graph = StateGraph(Blog_state)

graph.add_node('Make_outline',outline_draft)
graph.add_node('Make_Blog',Blog_draft)
graph.add_node('Blog_Evaluate',Evaluator)

graph.add_edge(START,'Make_outline')
graph.add_edge('Make_outline','Make_Blog')
graph.add_edge('Make_Blog',END)


In [11]:
workflow = graph.compile()

In [12]:
initial_state = {'title':'Indian Foods'}
final_state = workflow.invoke(initial_state)

print(final_state['title'])


BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

In [ ]:
print(final_state['outline'])

In [ ]:
print(final_state['blog'])

In [ ]:
print(final_state['score'])